1. UNet의 입출력을 픽셀공간이 아니라 잠재공간으로 축소 (이미지 픽셀 대신, z에 섞인 노이즈를 예측하게 하는 것)
2. 학습단계
  1. 이미지 데이터셋으로 VAE를 학습하여 VAE의 가중치 고정
  2. 학습된 VAE 인코더의 출력 z를 학습데이터로 하여 UNet을 학습
3. 생성은 z크기의 노이즈에서 시작하여, UNet으로 노이즈를 제거하고, VAE 디코더로 잠재공간으로부터 이미지를 생성
4. VAE의 출력인 z는 위치 구조가 남아있는 작은 이미지여야한다. UNet이 합성곱을 그대로 쓸 수 있도록 하기 위해.

### 02에서 가져온 부분
- `pos_encoding`, `ConvBlock`, `UNetCond`, `Diffuser`는 02와 같다.
- `sample`만 마지막의 `clamp(0, 1)`을 뺐다. 생성 결과가 이미지가 아니라 z이고, z의 값은 0~1 범위가 아니기 때문이다.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from tqdm import tqdm

NUM_CLASSES = 10
NULL_LABEL = NUM_CLASSES  # 레이블 0~9 다음 번호인 10을 '조건 없음' ∅로 쓴다.


def pos_encoding(ts, output_dim, device='cpu'):
    i = torch.arange(output_dim, device=device)
    div_term = 10000 ** (i / output_dim)
    angles = ts.to(device).float()[:, None] / div_term[None, :]
    v = torch.zeros(len(ts), output_dim, device=device)
    v[:, 0::2] = torch.sin(angles[:, 0::2])
    v[:, 1::2] = torch.cos(angles[:, 1::2])
    return v


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_embed_dim):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )
        self.mlp = nn.Sequential(
            nn.Linear(time_embed_dim, in_ch),
            nn.ReLU(),
            nn.Linear(in_ch, in_ch)
        )

    def forward(self, x, v):
        N, C, _, _ = x.shape
        v = self.mlp(v)
        v = v.view(N, C, 1, 1)
        y = self.convs(x + v)
        return y


class UNetCond(nn.Module):
    def __init__(self, in_ch=1, time_embed_dim=100, num_labels=None):
        super().__init__()
        self.time_embed_dim = time_embed_dim

        self.down1 = ConvBlock(in_ch, 64, time_embed_dim)
        self.down2 = ConvBlock(64, 128, time_embed_dim)
        self.bot1 = ConvBlock(128, 256, time_embed_dim)
        self.up2 = ConvBlock(256 + 128, 128, time_embed_dim)
        self.up1 = ConvBlock(128 + 64, 64, time_embed_dim)
        self.out = nn.Conv2d(64, in_ch, 1)

        self.maxpool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear')

        if num_labels is not None:
            self.label_emb = nn.Embedding(num_labels, time_embed_dim)

    def forward(self, x, timesteps, labels=None):
        t = pos_encoding(timesteps, self.time_embed_dim, device=x.device)
        if labels is not None:
            t += self.label_emb(labels)

        x1 = self.down1(x, t)
        x = self.maxpool(x1)
        x2 = self.down2(x, t)
        x = self.maxpool(x2)

        x = self.bot1(x, t)

        x = self.upsample(x)
        x = torch.cat([x, x2], dim=1)
        x = self.up2(x, t)
        x = self.upsample(x)
        x = torch.cat([x, x1], dim=1)
        x = self.up1(x, t)

        x = self.out(x)
        return x


class Diffuser:
    def __init__(self, num_timesteps=1000, beta_start=0.0001, beta_end=0.02, device='cpu'):
        self.num_timesteps = num_timesteps
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        self.alphas = 1 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)

    def add_noise(self, x_0, t):
        T = self.num_timesteps
        assert (t >= 1).all() and (t <= T).all()
        alpha_bar = self.alpha_bars[t - 1].view(-1, 1, 1, 1)
        noise = torch.randn_like(x_0, device=self.device)
        x_t = torch.sqrt(alpha_bar) * x_0 + torch.sqrt(1 - alpha_bar) * noise
        return x_t, noise

    # CFG 노이즈 예측 ε̃ = ε̂(x_t, ∅) + w·(ε̂(x_t, y) − ε̂(x_t, ∅))를 계산하고, guidance_scale이 w다.
    def guided_noise(self, model, x, t, labels, guidance_scale):
        null_labels = torch.full_like(labels, NULL_LABEL)
        # eval 상태의 BatchNorm은 누적 통계를 쓰므로, 조건부 입력과 무조건 입력을 한 배치로 묶어 UNet을 한 번만 호출해도 결과가 같다.
        eps = model(torch.cat([x, x]), torch.cat([t, t]), torch.cat([labels, null_labels]))
        eps_cond, eps_uncond = eps.chunk(2)
        return eps_uncond + guidance_scale * (eps_cond - eps_uncond)

    def denoise(self, model, x, t, labels, guidance_scale):
        T = self.num_timesteps
        assert (t >= 1).all() and (t <= T).all()

        t_idx = t - 1
        alpha = self.alphas[t_idx].view(-1, 1, 1, 1)
        alpha_bar = self.alpha_bars[t_idx].view(-1, 1, 1, 1)
        alpha_bar_prev = self.alpha_bars[t_idx - 1].view(-1, 1, 1, 1)

        model.eval()
        with torch.no_grad():
            eps = self.guided_noise(model, x, t, labels, guidance_scale)
        model.train()

        noise = torch.randn_like(x, device=self.device)
        noise[t == 1] = 0

        mu = (x - ((1 - alpha) / torch.sqrt(1 - alpha_bar)) * eps) / torch.sqrt(alpha)
        std = torch.sqrt((1 - alpha) * (1 - alpha_bar_prev) / (1 - alpha_bar))
        return mu + noise * std

    def sample(self, model, labels, guidance_scale, x_shape):
        batch_size = len(labels)
        x = torch.randn((batch_size, *x_shape), device=self.device)

        for i in tqdm(range(self.num_timesteps, 0, -1)):
            t = torch.full((batch_size,), i, device=self.device, dtype=torch.long)
            x = self.denoise(model, x, t, labels, guidance_scale)
        # z는 0~1 범위의 값이 아니므로 clamp하지 않고, 디코더를 거친 이미지만 0~1로 다룬다.
        return x

### 1단계: VAE 학습
- 28×28 이미지를 32×32로 패딩한다.
  - 인코더가 크기를 두 번 절반으로 줄이면 8×8이 되고, UNet도 8×8을 두 번 절반으로 줄였다가 되돌릴 수 있다.
  - 28×28을 그대로 쓰면 z가 7×7이 된다. UNet이 7을 절반으로 줄인 3을 다시 2배로 늘리면 6이 되어, skip connection과 크기가 맞지 않는다.
- 인코더는 32×32×1 이미지를 8×8×4 z로 줄인다. 값 개수는 1,024개에서 256개가 된다.
- step7과 달리 Linear 대신 합성곱을 써서, z가 위치 구조를 가진 작은 이미지가 되게 한다.
- KL 항의 가중치는 0.01이다. 1로 두면 잠재 채널 대부분이 입력과 관계없는 값이 되어 재구성 오차가 커진다. SD의 VAE도 KL 항의 가중치를 아주 작게 둔다.

In [ ]:
batch_size = 128
latent_ch = 4
vae_epochs = 5
vae_lr = 1e-3
kl_weight = 0.01
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device('cpu')

preprocess = transforms.Compose([transforms.Pad(2), transforms.ToTensor()])  # (1, 32, 32)
dataset = torchvision.datasets.MNIST(root='./data', download=True, transform=preprocess)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


class Encoder(nn.Module):
    def __init__(self, latent_ch):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),   # (N, 32, 16, 16)
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # (N, 64, 8, 8)
            nn.ReLU()
        )
        self.conv_mu = nn.Conv2d(64, latent_ch, 3, padding=1)      # (N, 4, 8, 8)
        self.conv_logvar = nn.Conv2d(64, latent_ch, 3, padding=1)  # (N, 4, 8, 8)

    def forward(self, x):
        h = self.convs(x)
        mu = self.conv_mu(h)
        sigma = torch.exp(0.5 * self.conv_logvar(h))
        return mu, sigma


class Decoder(nn.Module):
    def __init__(self, latent_ch):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(latent_ch, 64, 3, padding=1),              # (N, 64, 8, 8)
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),  # (N, 32, 16, 16)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1),   # (N, 1, 32, 32)
            nn.Sigmoid()
        )

    def forward(self, z):
        return self.convs(z)


def reparameterize(mu, sigma):
    eps = torch.randn_like(sigma)
    return mu + eps * sigma


class VAE(nn.Module):
    def __init__(self, latent_ch, kl_weight):
        super().__init__()
        self.encoder = Encoder(latent_ch)
        self.decoder = Decoder(latent_ch)
        self.kl_weight = kl_weight

    def get_loss(self, x):
        mu, sigma = self.encoder(x)
        z = reparameterize(mu, sigma)
        x_hat = self.decoder(z)

        L1 = F.mse_loss(x_hat, x, reduction='sum')
        L2 = -torch.sum(1 + torch.log(sigma ** 2) - mu ** 2 - sigma ** 2)
        return (L1 + self.kl_weight * L2) / len(x)


vae = VAE(latent_ch, kl_weight).to(device)
vae_optimizer = Adam(vae.parameters(), lr=vae_lr)

for epoch in range(vae_epochs):
    loss_sum = 0.0
    for images, _ in tqdm(dataloader):
        x = images.to(device)
        vae_optimizer.zero_grad()
        loss = vae.get_loss(x)
        loss.backward()
        vae_optimizer.step()
        loss_sum += loss.item()
    print(f'Epoch {epoch} | Loss: {loss_sum / len(dataloader)}')

### z 확인
- 원본, z의 채널 4개, z를 디코더로 복원한 이미지를 나란히 본다.
- z의 채널에도 숫자 윤곽이 남아 있으면, z가 위치 구조를 가진 작은 이미지라는 뜻이다.

In [ ]:
images = torch.stack([dataset[i][0] for i in range(6)]).to(device)
with torch.no_grad():
    mu, _ = vae.encoder(images)  # (6, 4, 8, 8)
    recon = vae.decoder(mu)      # (6, 1, 32, 32)

titles = ['x (32x32)', 'z[0] (8x8)', 'z[1]', 'z[2]', 'z[3]', 'decode(z)']
fig, axes = plt.subplots(len(images), len(titles), figsize=(len(titles), len(images)))
for r in range(len(images)):
    panels = [images[r, 0], *mu[r], recon[r, 0]]
    for c, panel in enumerate(panels):
        ax = axes[r, c]
        ax.imshow(panel.cpu(), cmap='gray')
        ax.set_xticks([])
        ax.set_yticks([])
        if r == 0:
            ax.set_title(titles[c], fontsize=8)
plt.tight_layout()
plt.show()

### 2단계: z로 UNet 학습
- VAE의 가중치를 고정하고, 데이터셋 전체를 한 번 인코딩해 z를 만든다. 가중치가 고정되어 있고 평균 mu를 z로 쓰므로, 에폭마다 다시 인코딩해도 같은 값이 나오기 때문이다.
- z에 `scale_factor`를 곱해 표준편차를 1로 맞춘다. `add_noise`의 노이즈 일정은 x_0의 크기가 노이즈와 비슷하다고 보고 정한 값이기 때문이다. SD의 0.18215도 같은 역할이다.
- 학습 코드는 02와 같고, 입력만 이미지 대신 z다. 그래서 UNet의 입력 채널이 1에서 4로 바뀐다.

In [ ]:
num_timesteps = 1000
epochs = 10
lr = 1e-3
p_uncond = 0.1

vae.requires_grad_(False)
with torch.no_grad():
    latents, latent_labels = [], []
    for images, labels in DataLoader(dataset, batch_size=1000):
        mu, _ = vae.encoder(images.to(device))
        latents.append(mu)
        latent_labels.append(labels)
latents = torch.cat(latents)                         # (60000, 4, 8, 8)
latent_labels = torch.cat(latent_labels).to(device)  # (60000,)

scale_factor = 1 / latents.std()
print(f'z std: {latents.std():.3f}, scale_factor: {scale_factor:.3f}')
latent_loader = DataLoader(TensorDataset(latents * scale_factor, latent_labels), batch_size=batch_size, shuffle=True)

diffuser = Diffuser(num_timesteps, device=device)
model = UNetCond(in_ch=latent_ch, num_labels=NUM_CLASSES + 1).to(device)
optimizer = Adam(model.parameters(), lr=lr)

losses = []
for epoch in range(epochs):
    loss_sum = 0.0
    cnt = 0

    for z, labels in tqdm(latent_loader):
        optimizer.zero_grad()
        is_dropped = torch.rand(len(labels), device=device) < p_uncond
        labels = torch.where(is_dropped, NULL_LABEL, labels)
        t = torch.randint(1, num_timesteps + 1, (len(z),), device=device)

        z_noisy, noise = diffuser.add_noise(z, t)
        noise_pred = model(z_noisy, t, labels)
        loss = F.mse_loss(noise, noise_pred)

        loss.backward()
        optimizer.step()

        loss_sum += loss.item()
        cnt += 1

    losses.append(loss_sum / cnt)
    print(f'Epoch {epoch} | Loss: {losses[-1]}')

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

### 생성
- 8×8×4 노이즈에서 시작해, UNet으로 1000단계 동안 노이즈를 제거한다.
- 마지막에 `scale_factor`로 나눠 원래 z의 크기로 되돌린 뒤, 디코더를 한 번 호출해 32×32 이미지로 바꾼다. 인코더는 생성에 쓰지 않는다.

In [ ]:
guidance_scale = 3.0
labels = torch.arange(NUM_CLASSES, device=device).repeat(2)

torch.manual_seed(0)
z = diffuser.sample(model, labels, guidance_scale, x_shape=(latent_ch, 8, 8))  # (20, 4, 8, 8)
with torch.no_grad():
    images = vae.decoder(z / scale_factor).cpu()                                # (20, 1, 32, 32)

fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(NUM_CLASSES, 2.4))
for ax, image, label in zip(axes.flat, images, labels):
    ax.imshow(image[0], cmap='gray')
    ax.set_title(label.item())
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()